In [1]:
import os
import pandas as pd
from maomao.parsing.parsing_utils import *
from maomao.utils.constants import *

#### Processing and standardizing peptide datasets (ToxMSRC)

This notebook processes and standardizes the **ToxMSRC** dataset into a consistent format suitable for downstream analysis and machine learning workflows. The source provides peptide sequences across multiple files and formats (FASTA and CSV), distributed into “Raw data” and “Example data” folders.

- **Toxic effect / endpoint:** toxic
- **Source:** ToxMSRC
- **Sequence scope:** only non-modified peptide sequences are retained for the final dataset.

The pipeline performs the following steps:

- **Loads peptide sequences from FASTA files** and derives labels from the FASTA record identifiers when available (e.g., `id|<label>`).
- **Loads peptide sequences from CSV files** containing sequence/label columns.
- **Concatenates all records** into a single DataFrame and enforces the canonical schema:
  - `sequence`: peptide amino-acid sequence (string)
  - `label`: toxicity label (integer)
- **Performs duplicate sequence quality control**:
  - keeps duplicated sequences only when their labels are consistent,
  - flags sequences with conflicting labels as erroneous.
- **Creates dataset-level metadata** using the centralized raw-data description spreadsheet.
- **Exports the curated dataset** and metadata to the standardized output directory.

In [2]:
name_source = "ToxMSRC"
name_task = "toxic_effect_classification"

# PATH_INPUT and PATH_EXPORT are imported from maomao.utils.constants
# Update them in constants.py according to the required input and export paths.

- Reading raw data

In [3]:
df_fasta = pd.concat([
    read_fasta_doc(os.path.join(folder, file))
    for folder in [f"{PATH_INPUT}/{name_source}/Raw data", 
                   f"{PATH_INPUT}/{name_source}/Example data"] 
    for file in os.listdir(folder)
    if file.endswith(".fasta")
])

In [4]:
df_csv = pd.concat([
    pd.read_csv(os.path.join(folder, file), skiprows=1, names=["sequence", "label"])
    for folder in [f"{PATH_INPUT}/{name_source}/Raw data", 
                   f"{PATH_INPUT}/{name_source}/Example data"] 
    for file in os.listdir(folder)
    if file.endswith(".csv")
])

In [5]:
df_fasta = (
    df_fasta.assign(
        label=df_fasta["id"].str.split("|").str[1].astype(int)
    )
    [["sequence", "label"]]
)

- Concatenate dataset

In [6]:
df_toxmsrc = (
    pd.concat([df_fasta, df_csv], ignore_index=True)
    .assign(label=1)
    [["sequence", "label"]]
)
df_toxmsrc.shape

(8999, 2)

- Checking duplicates

In [7]:
df_remove_duplicated, df_errors, df_unique = processing_duplicated(df_toxmsrc, group_seq="sequence", sort_key="label")
df_full = pd.concat([df_unique, df_remove_duplicated], axis=0)

In [8]:
df_full.shape

(7513, 2)

In [9]:
df_errors.shape

(0, 1)

- Working with metada

In [10]:
df_metada = read_metadata("../../raw_data/raw_data_description.xlsx", name_source)
dict_metadata = create_metada_with_multiple_values(df_metada)

In [11]:
dict_metadata.update({
    "number_of_raw_sequences": int(len(df_toxmsrc)),
    "number_of_sequences_retained": len(df_full),
    "number_of_positive_sequences": int((df_full["label"] == 1).sum()),
    "number_of_negative_sequences": int((df_full["label"] == 0).sum()),
    "number_of_erroneous_sequences": int(len(df_errors)),
    "modified_sequences_included": False,
})

dict_metadata

{'type source': 'Dataset',
 'static-dynamic': 'Static',
 'license': 'No information',
 'year of publication': 2025,
 'last update date': datetime.datetime(2025, 8, 25, 0, 0),
 'download date': Timestamp('2025-10-17 00:00:00'),
 'file format': 'csv;fasta',
 'peptide property': 'toxic',
 'dataset information': 'Positive, Negative',
 'unit of measurement': 'No information',
 'obtaining negative dataset': 'Sampling from another DB',
 'repository or server': 'https://github.com/Renjingyi123/ToxMSRC/tree/main/Raw%20data',
 'publication': 'https://academic.oup.com/bioinformatics/advance-article/doi/10.1093/bioinformatics/btaf462/8239951',
 'number_of_raw_sequences': 8999,
 'number_of_sequences_retained': 7513,
 'number_of_positive_sequences': 7513,
 'number_of_negative_sequences': 0,
 'number_of_erroneous_sequences': 0,
 'modified_sequences_included': False}

- Exporting data

In [12]:
os.makedirs(f"{PATH_EXPORT}/{name_task}/{name_source}/", exist_ok=True)
export_json(f"{PATH_EXPORT}/{name_task}/{name_source}/metadata.json", dict_metadata)

In [13]:
df_full.to_csv(f"{PATH_EXPORT}/{name_task}/{name_source}/processed_toxic_dataset.csv", index=False)